In [ ]:
import sglang as sgl
import nest_asyncio
nest_asyncio.apply()
 
# Cell 1: Start engine
engine = sgl.Engine(model_path="Qwen/Qwen3.5-0.8B", mem_fraction_static=0.7)


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")

messages = [{"role": "user", "content": "Hello, how are you?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

response = engine.generate(prompt=prompt, sampling_params={"max_new_tokens": 100})
print(response["text"])

Hello! I'm Avatar, a helpful AI assistant. How can I help you today? 😊


In [10]:

# Final cell: Shutdown
engine.shutdown()

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3.5-0.8B"

# 1. Load the Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

# 2. Prepare the input using a Chat Template
messages = [{"role": "user", "content": "Explain what a tensor is in one sentence."}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 3. Tokenize the text (convert to numbers)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# 4. Generate output IDs
outputs = model.generate(
    **inputs, 
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True
)

# 5. Decode the IDs back to text
# We slice [0] because generate returns a batch, and we skip the input tokens
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

print(response)

/home/alvin/experiments/projects/mech_interp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 182.61it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


A tensor is a mathematical object that can be represented as a matrix of tensors.



In [11]:
type(model.model.embed_tokens)

torch.nn.modules.sparse.Embedding

In [7]:
layer_0 = model.model.layers[0]
print(layer_0)

Qwen3_5DecoderLayer(
  (linear_attn): Qwen3_5GatedDeltaNet(
    (act): SiLUActivation()
    (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
    (norm): Qwen3_5RMSNormGated()
    (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (in_proj_qkv): Linear(in_features=1024, out_features=6144, bias=False)
    (in_proj_z): Linear(in_features=1024, out_features=2048, bias=False)
    (in_proj_b): Linear(in_features=1024, out_features=16, bias=False)
    (in_proj_a): Linear(in_features=1024, out_features=16, bias=False)
  )
  (mlp): Qwen3_5MLP(
    (gate_proj): Linear(in_features=1024, out_features=3584, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3584, bias=False)
    (down_proj): Linear(in_features=3584, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3_5RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3_5RMSNorm((1024,), eps=1e-06)
)


In [5]:
for name, module in model.named_modules():
    print(name, type(module).__name__)

 Qwen3_5ForCausalLM
model Qwen3_5TextModel
model.embed_tokens Embedding
model.layers ModuleList
model.layers.0 Qwen3_5DecoderLayer
model.layers.0.linear_attn Qwen3_5GatedDeltaNet
model.layers.0.linear_attn.act SiLUActivation
model.layers.0.linear_attn.conv1d Conv1d
model.layers.0.linear_attn.norm Qwen3_5RMSNormGated
model.layers.0.linear_attn.out_proj Linear
model.layers.0.linear_attn.in_proj_qkv Linear
model.layers.0.linear_attn.in_proj_z Linear
model.layers.0.linear_attn.in_proj_b Linear
model.layers.0.linear_attn.in_proj_a Linear
model.layers.0.mlp Qwen3_5MLP
model.layers.0.mlp.gate_proj Linear
model.layers.0.mlp.up_proj Linear
model.layers.0.mlp.down_proj Linear
model.layers.0.mlp.act_fn SiLUActivation
model.layers.0.input_layernorm Qwen3_5RMSNorm
model.layers.0.post_attention_layernorm Qwen3_5RMSNorm
model.layers.1 Qwen3_5DecoderLayer
model.layers.1.linear_attn Qwen3_5GatedDeltaNet
model.layers.1.linear_attn.act SiLUActivation
model.layers.1.linear_attn.conv1d Conv1d
model.layers.

# sglang

In [ ]:
# # call sglang server for inference

# import requests

# url = "http://127.0.0.1:30000/v1/chat/completions"

# payload = {
#     "model": "default",  # SGLang ignores this for single-model setups
#     "messages": [
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "Explain quantum computing in one sentence."}
#     ],
#     "temperature": 0.6,
#     "top_p": 0.95,
#     "max_tokens": 256
# }

# # response = requests.post(url, json=payload)
# # print(response.json()["choices"][0]["message"]["content"])

In [ ]:
# import json

# payload["stream"] = True
# response = requests.post(url, json=payload, stream=True)

# for line in response.iter_lines():
#     if line:
#         text = line.decode("utf-8").removeprefix("data: ")
#         if text == "[DONE]":
#             break
#         chunk = json.loads(text)
#         print(chunk["choices"][0]["delta"].get("content", ""), end="")

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=30000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=30000): Failed to establish a new connection: [Errno 111] Connection refused"))